In [3]:
import cv2
import numpy as np
import mediapipe as mp

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

## Portrait Mode <small>(webcam)</small>

In [4]:
cap = cv2.VideoCapture(0)

with mp_pose.Pose(
    static_image_mode=False, 
    model_complexity=1,   # 실시간 웹캠 사용시
    enable_segmentation=True, 
    min_detection_confidence=0.5
) as pose:

    while(cap.isOpened()):
        a, frame = cap.read()
        frame = cv2.resize(frame, (640, 360))
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image_height, image_width, _ = image.shape
        results = pose.process(image)
        annotated_image = image.copy()

        if results.segmentation_mask is not None: # 사람 유무 확인
            condition = np.stack((results.segmentation_mask,) * 3, axis=-1) > 0.1
            bg_image = np.zeros(image.shape, dtype=np.uint8)
            blur_image = image.copy()

            for i in range(2):
                blur_image = cv2.GaussianBlur(blur_image, (17,17), 0)
                annotated_image = np.where(condition, annotated_image, blur_image)

        '''
        # 사람 위에 스켈레톤으로 보여줌
        if results.pose_landmarks is not None: # 사람 
            mp_drawing.draw_landmarks(annotated_image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())
        '''

        annotated_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

        cv2.imshow("img", annotated_image)

        k = cv2.waitKey(15) & 0xFF
        if k == 27: # ESC
            break

cv2.destroyAllWindows()